# Import libs

In [1]:
from __future__ import print_function

import os
import time

import cv2
import numpy as np

import torch
import torch.nn.functional as F
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights

from piqa import SSIM

from models.skip import skip
from utils.inpainting_utils import *

torch.backends.cudnn.enabled = True
torch.backends.cudnn.benchmark = True
dtype = torch.cuda.FloatTensor

imsize = -1

# Setup

In [2]:
class SSIMLoss(SSIM):
    def forward(self, x, y):
        return 1. - super().forward(x, y)

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
weights = Raft_Large_Weights.DEFAULT
model_raft = raft_large(weights=weights).to(device)
model_raft.eval()

def create_flow(frame1, frame2):
    # возвращает поток при переходе из frame1 в frame2
    
    frame1_norm = frame1 * 2 - 1
    frame2_norm = frame2 * 2 - 1
    with torch.no_grad():
        flow_predictions = model_raft(frame2_norm,frame1_norm)
    flow = flow_predictions[-1] 
    return flow

def warp_with_flow(frame_prev, flow):
    """
    frame_prev: [B, C, H, W]
    flow: [B, 2, H, W]  (dx, dy) в пикселях

    возвращает warped frame: [B, C, H, W]
    """
    B, C, H, W = frame_prev.shape

    y, x = torch.meshgrid(
        torch.arange(H, device=frame_prev.device),
        torch.arange(W, device=frame_prev.device),
        indexing="ij"
    )

    grid = torch.stack((x, y), dim=0).float()  # [2,H,W]
    grid = grid.unsqueeze(0).repeat(B, 1, 1, 1)  # [B,2,H,W]

    vgrid = grid + flow

    vgrid[:, 0] = 2.0 * vgrid[:, 0] / (W - 1) - 1.0
    vgrid[:, 1] = 2.0 * vgrid[:, 1] / (H - 1) - 1.0

    vgrid = vgrid.permute(0, 2, 3, 1)

    warped = F.grid_sample(
        frame_prev,
        vgrid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True
    )

    return warped

def inpaint_flow_tensor(flow, mask):
    """
    flow: [2,H,W] torch.float32
    mask: [1,H,W] torch.uint8, 0 = закрашено, 1 = валидные пиксели

    возвращает flow: [2,H,W] torch.float32 с заполненной маской
    """
    flow_np = flow.cpu().numpy()  # [2,H,W]
    mask_np = mask[0].cpu().numpy()  # [H,W]

    flow_out = flow_np.copy()

    inpaint_mask = (mask_np == 0).astype(np.uint8) * 255

    for i in range(2):
        channel = flow_np[i].astype(np.float32)
        inpainted = cv2.inpaint(channel, inpaint_mask, 5, cv2.INPAINT_TELEA)
        flow_out[i] = inpainted

    flow_out_tensor = torch.from_numpy(flow_out).to(flow.device)
    return flow_out_tensor.unsqueeze(0)

In [4]:
class OptimizerWrapper:
    def __init__(self, net, net_input, mask_var,path):
        self.net = net
        self.mask_var = mask_var
        self.ssim = SSIMLoss().type(dtype)
        self.l1 = torch.nn.L1Loss().type(dtype)
        self.reg_noise_std = 0.03
        self.net_input = net_input
        self.img_var = None
        self.save_loss = []
        self.iter_save=[]
        self.last_warped = None
        self.last_img_var = None
        self.last_out = None
        self.noise = torch.zeros_like(net_input)
        self.i=0
        self.masked_img = None
        self.masked_warped = None
        self.imfirst=True
        self.path = path
        self.scaler = torch.amp.GradScaler("cuda")

    def get_result(self, img_var, textt, num_iter=5001):
        """
        img_var — текущий кадр в виде тензора [B, C, H, W]
        textt — имя файла для сохранения результата
        num_iter — максимальное количество итераций обучения

        не возвращает значения; сохраняет восстановленный кадр
        """
        
        self.i=0
        self.img_var = img_var
        self.masked_img = self.img_var * self.mask_var
        p = get_params('net', self.net, self.net_input)
        self.save_loss=[]
        if self.imfirst:
            self.optimize(p, 0.01, num_iter)
            self.imfirst = False
        else:
            flow = create_flow(self.last_img_var * self.mask_var, self.masked_img)
            inp_flow = inpaint_flow_tensor(flow[0], self.mask_var[0])
            self.last_warped = warp_with_flow(self.last_out, inp_flow).type(dtype)
            self.masked_warped = self.last_warped[0] * (1 - self.mask_var[0])
            self.optimize(p, 0.01, num_iter)
        with torch.no_grad():
          out = self.net(self.net_input)
        self.last_out = out.detach().clone()
        self.last_img_var = img_var.clone()
        out_np = torch_to_np(self.last_out)
        np_to_pil(out_np).save(self.path+"/"+textt+".png")
        self.iter_save.append(len(self.save_loss))

    def optimize(self,parameters, LR, num_iter):
        """
        parameters — параметры сети
        LR — скорость обучения
        num_iter — максимальное количество итераций

        не возвращает значения; обучает нейронную сеть
        """
        
        optimizer = torch.optim.Adam(parameters, lr=LR)
        for j in range(num_iter):
            optimizer.zero_grad(set_to_none=True)
            self.closure()
            if not self.do_cont():
                break
            self.scaler.step(optimizer)
            self.scaler.update()

    def closure(self):
        #возвращает значение лосс-функции для текущей итерации
        
        self.net_input += (self.noise.normal_() * self.reg_noise_std)
        with torch.amp.autocast("cuda"):
          out = self.net(self.net_input)
        out = out.float()
        if self.imfirst:
            total_loss = self.ssim(out * self.mask_var, self.masked_img)
        else:
            total_loss = self.ssim(out * self.mask_var, self.masked_img) + 0.1*self.l1(out[0]*(1-self.mask_var[0]),self.masked_warped)
        #print ('Iteration %05d    Loss %f' % (self.i, total_loss.item()), '\r', end='')
        self.save_loss.append(total_loss.item())
        self.scaler.scale(total_loss).backward()
        self.i+=1
        return total_loss

    def do_cont(self,num=100,eps=0.01):
        """
        num — минимальное число итераций для кадра
        eps — порог изменения лосс-функции, после которого изменения считаются незначительными

        возвращает True, если обучение следует продолжить,
                   False, иначе.
        """
        
        if self.imfirst:
            num=200
        if len(self.save_loss)>num:
            an = np.array(self.save_loss[-num:])
            ans = np.max(an) - np.min(an)
            if ans<eps:
                return False
            return True
        return True

# Example for DAVIS 2017

In [6]:
#словарь - название папки: число кадров

dictt = {"bear": 82, "bike-packing": 69, "blackswan": 50, "bmx-bumps": 90, "bmx-trees": 80, "boat": 75, "boxing-fisheye": 87, "breakdance": 84, "breakdance-flare": 71, "bus": 80, "camel": 90, "car-roundabout": 75, "car-shadow": 40, "car-turn": 80, "cat-girl": 89, "classic-car": 63, "color-run": 84, "cows": 104, "crossing": 52, "dance-jump": 60, "dance-twirl": 90, "dancing": 62, "disc-jockey": 76, "dog": 60, "dog-agility": 25, "dog-gooses": 86, "dogs-jump": 66, "dogs-scale": 83, "drift-chicane": 52, "drift-straight": 50, "drift-turn": 64, "drone": 91, "elephant": 80, "flamingo": 80, "goat": 90, "gold-fish": 78, "hike": 80, "hockey": 75, "horsejump-high": 50, "horsejump-low": 60, "india": 81, "judo": 34, "kid-football": 68, "kite-surf": 50, "kite-walk": 80, "koala": 100, "lab-coat": 47, "lady-running": 65, "libby": 49, "lindy-hop": 73, "loading": 50, "longboard": 52, "lucia": 70, "mallard-fly": 70, "mallard-water": 80, "mbike-trick": 79, "miami-surf": 70, "motocross-bumps": 60, "motocross-jump": 40, "motorbike": 43, "night-race": 46, "paragliding": 70, "paragliding-launch": 80, "parkour": 100, "pigs": 79, "planes-water": 38, "rallye": 50, "rhino": 90, "rollerblade": 35, "schoolgirls": 80, "scooter-black": 43, "scooter-board": 91, "scooter-gray": 75, "sheep": 68, "shooting": 40, "skate-park": 80, "snowboard": 66, "soapbox": 99, "soccerball": 48, "stroller": 91, "stunt": 71, "surf": 55, "swing": 60, "tennis": 70, "tractor-sand": 76, "train": 80, "tuk-tuk": 59, "upside-down": 65, "varanus-cage": 67, "walking": 72}

In [7]:
list_frames = list(dictt.values())

In [ ]:
pad = 'reflection'
INPUT = 'meshgrid'
input_depth = 2

for folder in range(1,91):
    start = time.perf_counter()
    
    folder_path = 'data/videos/video'+str(folder) #путь папки с кадрами видео для восстановления
    mask_path = folder_path+'/mask.png'
    out_path = 'data/results/video'+str(folder) #путь папки, в которую следует сохранять результаты
    os.makedirs(out_path, exist_ok=True)
    
    img_mask_pil, img_mask_np = get_image(mask_path, imsize)
    mask_var = np_to_torch(img_mask_np).type(dtype)
    img_path = folder_path+f"/frame_{1:04d}_gt.png"
    img_pil, img_np = get_image(img_path, imsize)
    net = skip(input_depth, img_np.shape[0],
               num_channels_down = [128] * 5,
               num_channels_up   = [128] * 5,
               num_channels_skip = [0] * 5,  
               upsample_mode='nearest', filter_skip_size=1, filter_size_up=3, filter_size_down=3,
               need_sigmoid=True, need_bias=True, pad=pad, act_fun="Soft").type(dtype)
    net = net.type(dtype)
    net_input = get_noise(input_depth, INPUT, img_np.shape[1:]).type(dtype)
    my_opt = OptimizerWrapper(net, net_input, mask_var,out_path)

    for cadr in range(list_frames[folder-1]):
        img_path = folder_path+f"/frame_{cadr:04d}_gt.png"
        img_pil, img_np = get_image(img_path, imsize)
        img_var = np_to_torch(img_np).type(dtype)
        textt = f'frame_{cadr:04d}_pred'
        my_opt.get_result(img_var=img_var,textt=textt)
    
    end = time.perf_counter()
    print(folder, f"Время выполнения: {end - start:.4f} секунд")

# Example for any video

In [ ]:
pad = 'reflection'
INPUT = 'meshgrid'
input_depth = 2

start = time.perf_counter()
    
folder_path = 'data/videos/video1'          #путь папки с кадрами видео для восстановления
num_frames = 81                             #число кадров видео
mask_path = folder_path+'/mask.png'         #путь маски восстановления
out_path = 'data/results/video'+str(folder) #путь папки, в которую следует сохранять результаты
os.makedirs(out_path, exist_ok=True)
    
img_mask_pil, img_mask_np = get_image(mask_path, imsize)
mask_var = np_to_torch(img_mask_np).type(dtype)
img_path = folder_path+f"/frame_{1:04d}_gt.png" #путь первого кадра
img_pil, img_np = get_image(img_path, imsize)
net = skip(input_depth, img_np.shape[0],
               num_channels_down = [128] * 5,
               num_channels_up   = [128] * 5,
               num_channels_skip = [0] * 5,  
               upsample_mode='nearest', filter_skip_size=1, filter_size_up=3, filter_size_down=3,
               need_sigmoid=True, need_bias=True, pad=pad, act_fun="Soft").type(dtype)
net = net.type(dtype)
net_input = get_noise(input_depth, INPUT, img_np.shape[1:]).type(dtype)
my_opt = OptimizerWrapper(net, net_input, mask_var,out_path)

for cadr in range(num_frames):
    img_path = folder_path+f"/frame_{cadr:04d}_gt.png" #путь кадра с №cadr
    img_pil, img_np = get_image(img_path, imsize)
    img_var = np_to_torch(img_np).type(dtype)
    textt = f'frame_{cadr:04d}_pred' #название исправленного кадра
    my_opt.get_result(img_var=img_var,textt=textt)
    
end = time.perf_counter()
print(folder, f"Время выполнения: {end - start:.4f} секунд")